In [ ]:
import pandas as pd

# 1. Exact paths to your folders where the raw data files are stored
chemin_ref = r"Z:\SAE\SAE15\ref\ref"
chemin_data = r"Z:\SAE\SAE15\data\data"

print("Loading all reference and data files...")
# Load reference tables (containing names/labels for types, owners, nature, and operators)
type_antenne = pd.read_csv(f"{chemin_ref}\\sup_type_antenne.txt", sep=";", low_memory=False)
proprietaire = pd.read_csv(f"{chemin_ref}\\sup_proprietaire.txt", sep=";", low_memory=False)
nature = pd.read_csv(f"{chemin_ref}\\sup_nature.txt", sep=";", low_memory=False)
exploitant = pd.read_csv(f"{chemin_ref}\\sup_exploitant.txt", sep=";", low_memory=False)

# Load main data tables (antennas, stations, and supports/towers)
antenne = pd.read_csv(f"{chemin_data}\\sup_antenne.txt", sep=";", low_memory=False)
station = pd.read_csv(f"{chemin_data}\\sup_station.txt", sep=";", low_memory=False)
suport = pd.read_csv(f"{chemin_data}\\sup_support.txt", sep=";", low_memory=False)

# 2. Keep only the useful columns we need to save memory and process faster
antenne = antenne[['STA_NM_ANFR', 'TAE_ID']]
station = station[['STA_NM_ANFR', 'ADM_ID', 'DTE_EN_SERVICE']]
suport = suport[['STA_NM_ANFR', 'TPO_ID', 'NAT_ID', 'SUP_NM_HAUT', 
                 'COR_NB_DG_LAT', 'COR_NB_MN_LAT', 'COR_CD_NS_LAT', 'COR_NB_SC_LAT', 
                 'COR_NB_DG_LON', 'COR_NB_MN_LON', 'COR_NB_SC_LON', 'COR_CD_EW_LON',
                 'ADR_NM_CP', 'ADR_LB_LIEU']]

print("Merging all tables together...")
# Merge tables one by one using common IDs (like station name, operator ID, owner ID, etc.)
ss = pd.merge(suport, station, on='STA_NM_ANFR', how='left')
ssa = pd.merge(ss, antenne, on='STA_NM_ANFR', how='left')
ssae = pd.merge(ssa, exploitant, on='ADM_ID', how='left')
ssaen = pd.merge(ssae, nature, on='NAT_ID', how='left')
ssaenp = pd.merge(ssaen, proprietaire, on='TPO_ID', how='left')
ssaenpt = pd.merge(ssaenp, type_antenne, on='TAE_ID', how='left')

print("Converting geographic coordinates from DMS (Degrees/Minutes/Seconds) to Decimal Degrees...")
# Select coordinate columns to ensure they are numbers
cols_num = ['COR_NB_DG_LAT', 'COR_NB_MN_LAT', 'COR_NB_SC_LAT',
            'COR_NB_DG_LON', 'COR_NB_MN_LON', 'COR_NB_SC_LON']
ssaenpt[cols_num] = ssaenpt[cols_num].apply(pd.to_numeric, errors='coerce')

# Convert Latitude (North = positive 1, South = negative -1)
dir_lat = ssaenpt['COR_CD_NS_LAT'].str.strip().str.upper().map({'N': 1, 'S': -1})
ssaenpt['latitude'] = (ssaenpt['COR_NB_DG_LAT'] + ssaenpt['COR_NB_MN_LAT'].fillna(0) / 60 + ssaenpt['COR_NB_SC_LAT'].fillna(0) / 3600) * dir_lat

# Convert Longitude (East = positive 1, West/West-OUEST = negative -1)
dir_lon = ssaenpt['COR_CD_EW_LON'].str.strip().str.upper().map({'E': 1, 'W': -1, 'O': -1})
ssaenpt['longitude'] = (ssaenpt['COR_NB_DG_LON'] + ssaenpt['COR_NB_MN_LON'].fillna(0) / 60 + ssaenpt['COR_NB_SC_LON'].fillna(0) / 3600) * dir_lon

# Round coordinates to 6 decimal places for cleanliness
ssaenpt[['latitude', 'longitude']] = ssaenpt[['latitude', 'longitude']].round(6)

# Remove any rows that do not have valid coordinates (latitude or longitude is missing)
ssaenpt = ssaenpt.dropna(subset=['latitude', 'longitude'])

# Clean text columns: replace empty spaces or missing values with 'Inconnu' (Unknown)
colonnes_texte = ssaenpt.select_dtypes(include=['object', 'string']).columns
ssaenpt[colonnes_texte] = ssaenpt[colonnes_texte].replace(r'^\s*$', 'Inconnu', regex=True).fillna('Inconnu')

# Save the fully cleaned and merged DataFrame into a clean CSV file (acts as a cache)
ssaenpt.to_csv("antenne_avec_coords.csv", sep=";", index=False, encoding="utf-8")
print("✅ File 'antenne_avec_coords.csv' successfully created with all details!")